In [28]:
def mostrar_reglas_legibles(rules_df):
    """
    Muestra reglas de asociación en formato legible.
    Convierte sets en texto y redondea columnas clave si existen.
    """
    df = rules_df.copy()
    
    # Convertir conjuntos a strings legibles
    df['antecedents'] = df['antecedents'].apply(lambda x: ', '.join(sorted(list(x))))
    df['consequents'] = df['consequents'].apply(lambda x: ', '.join(sorted(list(x))))

    # Columnas clave (si existen)
    columnas = ['support', 'confidence', 'lift']
    columnas_existentes = [col for col in columnas if col in df.columns]
    
    # Redondear columnas numéricas existentes
    df[columnas_existentes] = df[columnas_existentes].round(2)

    return df[['antecedents', 'consequents'] + columnas_existentes]


# EJEMPLO DE IMPLEMENTACIÓN
# _________________________________________________________

from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import pandas as pd

# Transacciones
dataset = [
          ['leche', 'manteca'],       
          ['leche', 'manteca'],       
          ['leche'],                  
          ['leche'],                  
          ['pan', 'manteca']
]

# Codificar transacciones / # Transformar
te = TransactionEncoder()
te_ary = te.fit(dataset).transform(dataset)
df = pd.DataFrame(te_ary, columns=te.columns_)

# Apriori Algorithm
frequent_itemsets = apriori(df, min_support=0.01, use_colnames=True) # Obtener itemsets frecuentes
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.01) # Obtener reglas

# Mostrar reglas legibles
reglas_legibles = mostrar_reglas_legibles(rules)
print(reglas_legibles)

  antecedents consequents  support  confidence  lift
0       leche     manteca      0.4         0.5   0.8
1     manteca       leche      0.4         0.7   0.8
2     manteca         pan      0.2         0.3   1.7
3         pan     manteca      0.2         1.0   1.7


<small>

**1. Regla: leche → manteca**

        Interpretación:
            En el 40% de las transacciones hay leche y manteca juntas.
            Cuando hay leche, en el 50% de los casos también hay manteca.
            El lift = 0.8, que es menor que 1, indica que:
                La relación es débil o independiente.
                Tener leche no incrementa la probabilidad de tener manteca (de hecho, es un poco menos probable).

        ✅ Este es exactamente el caso que querías ilustrar.

**2. Regla: manteca → leche** 

        Interpretación:
            Leche y manteca aparecen juntas el 40% de las veces.
            Cuando hay manteca, en el 70% de los casos hay leche.
            Lift = 0.8 → igual que antes, la relación es débil.

        📌 Aunque la confianza es más alta que en la regla anterior, el lift sigue siendo < 1, 
        por lo tanto, la dependencia no es significativa.  

**3. Regla: manteca → pan**

        Interpretación:
            Solo el 20% de las transacciones contienen manteca y pan juntas.
            Cuando hay manteca, en el 30% de los casos también hay pan.
            Pero el lift = 1.7, lo cual indica:
                Aunque esta regla ocurre poco, sí hay una fuerte asociación entre manteca y pan.
                Tener manteca incrementa notablemente la probabilidad de que también haya pan.
        🟡 Es una regla poco frecuente, pero con alto valor predictivo.  

**4. Regla: pan → manteca**

        Interpretación:
            En el 20% de las transacciones hay pan y manteca.
            Cuando hay pan, siempre hay manteca (confidence = 1.0).
            Lift = 1.7 → relación fuerte y positiva.
        ✅ Esta es una regla excelente si querés hacer recomendaciones:
        “Si hay pan, es muy probable que también haya manteca.”

<small>

###  **Conclusión**

| Regla              | ¿Útil? | ¿Por qué?                                       |
|--------------------|--------|------------------------------------------------|
| leche → manteca    | ❌     | Relación débil (lift < 1)                      |
| manteca → leche    | ❌     | Relación débil (lift < 1)                      |
| manteca → pan      | 🟡     | Relación fuerte pero poco frecuente            |
| pan → manteca      | ✅     | Alta confianza y fuerte relación (lift > 1)    |

¿Por qué pan → manteca y manteca → pan pueden tener resultados distintos?

Aunque los ítems sean los mismos, el orden importa muchísimo en reglas de asociación, porque no estamos midiendo una co-ocurrencia simétrica, sino una implicación direccional:

A → B  ≠  B → A

Porque lo que se mide es:

    confidence(A → B) = ¿Qué tan seguido aparece B cuando hay A?

    confidence(B → A) = ¿Qué tan seguido aparece A cuando hay B?

Y eso puede ser completamente distinto.

In [30]:
print (rules)

  antecedents consequents  antecedent support  consequent support  support  \
0     (leche)   (manteca)                 0.8                 0.6      0.4   
1   (manteca)     (leche)                 0.6                 0.8      0.4   
2   (manteca)       (pan)                 0.6                 0.2      0.2   
3       (pan)   (manteca)                 0.2                 0.6      0.2   

   confidence  lift  representativity  leverage  conviction  zhangs_metric  \
0         0.5   0.8               1.0      -0.1         0.8           -0.5   
1         0.7   0.8               1.0      -0.1         0.6           -0.3   
2         0.3   1.7               1.0       0.1         1.2            1.0   
3         1.0   1.7               1.0       0.1         inf            0.5   

   jaccard  certainty  kulczynski  
0      0.4       -0.2         0.6  
1      0.4       -0.7         0.6  
2      0.3        0.2         0.7  
3      0.3        1.0         0.7  
